# Papers: Digital Humanities Analysis

Extract, analyze, and visualize mathematical papers as structured knowledge graphs.

**Perspective:** Papers are not just PDFs—they are structured texts with theorems, lemmas, proofs, and citations. Treat them like MusicXML documents: machine-readable formal objects amenable to analysis, visualization, and extraction.

**Scope:**
- Extract metadata (authors, year, arXiv ID, title)
- Identify theorem/lemma structures
- Build citation networks
- Visualize proof dependencies
- Generate summary cards for website integration

---

## 1. Paper Metadata Registry

Structured database of all papers in the project.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import os

# Paper registry: mapping arXiv IDs to metadata and roles
PAPERS = {
    "1111.0931": {
        "arxiv_id": "1111.0931",
        "authors": ["Bettin", "Conrey"],
        "year": 2013,
        "title": "Approximations to the Riemann Hypothesis",
        "role": "H15 kernel machinery (period functions, cotangent sums)",
        "key_theorems": ["Period functional equation", "Vasyunin cotangent formula", "Bettin-Conrey reciprocity"],
        "status": "core",
        "notes": "Contains reciprocity machinery but not uniform Möbius-sawtooth bound"
    },
    "1601.06839": {
        "arxiv_id": "1601.06839",
        "authors": ["Auli", "Bayad", "Beck"],
        "year": 2017,
        "title": "Reciprocal Relations and *-Type Theorems",
        "role": "H15 reciprocity structure for cotangent sums",
        "key_theorems": ["Reciprocity for cotangent sums with gcd", "Integral representation (⚠️ stricter assumptions)"],
        "status": "core",
        "notes": "Stricter assumptions (Re(a)>1) than claimed in some references"
    },
    "1211.5191": {
        "arxiv_id": "1211.5191",
        "authors": ["Bettin", "Conrey", "Farmer"],
        "year": 2012,
        "title": "Explicit Formula for the Asymptotic Variance",
        "role": "Nyman-Beurling criterion and RH connection",
        "key_theorems": ["Nyman-Beurling machinery"],
        "status": "circular_logic",
        "notes": "⚠️ ASSUMES RH in main theorem—cannot use unconditionally to derive RH"
    },
    "1401.2932": {
        "arxiv_id": "1401.2932",
        "role": "H15 diagnostics: Farey cell binning",
        "status": "extended_collection"
    },
    "1401.3150": {
        "arxiv_id": "1401.3150",
        "role": "H15 Route 3: Dirichlet polynomial framework",
        "status": "extended_collection"
    },
    "1409.1634": {
        "arxiv_id": "1409.1634",
        "role": "H15 quadratic interaction kernels",
        "status": "extended_collection"
    },
    "1501.02975": {
        "arxiv_id": "1501.02975",
        "role": "H14 contour analysis; Route 1",
        "status": "extended_collection"
    },
    "1503.05121": {
        "arxiv_id": "1503.05121",
        "role": "Phase NB: critical line machinery",
        "status": "extended_collection"
    },
    "1510.05087": {
        "arxiv_id": "1510.05087",
        "role": "H14/H15: Estermann/Gamma machinery",
        "status": "extended_collection"
    },
    "math_0306251": {
        "arxiv_id": "math/0306251",
        "role": "H13 cotangent sum bounds (elementary proofs)",
        "status": "extended_collection",
        "notes": "Real source for tractable elementary proof chain"
    },
    "1807.08249": {
        "arxiv_id": "1807.08249",
        "role": "Phase NB: Hilbert space closure",
        "status": "extended_collection"
    },
    "2105.00004": {
        "arxiv_id": "2105.00004",
        "role": "Overall context and recent RH approaches",
        "status": "extended_collection"
    },
    "2209.10990": {
        "arxiv_id": "2209.10990",
        "role": "Phase NB: contemporary formulation",
        "status": "extended_collection"
    },
    "math_9912107": {
        "arxiv_id": "math/9912107",
        "role": "H14 linear estimates; zero-density results",
        "status": "extended_collection"
    }
}

# Convert to DataFrame for analysis
papers_df = pd.DataFrame.from_dict(PAPERS, orient='index')
print(f"Total papers registered: {len(papers_df)}")
print(f"Core papers (peer-reviewed): {len(papers_df[papers_df['status']=='core'])}")
print(f"Extended collection: {len(papers_df[papers_df['status']=='extended_collection'])}")
print("\n" + "="*80)
print(papers_df[['arxiv_id', 'role', 'status']].to_string())

## 2. Paper-Problem Alignment Matrix

Which papers are relevant to which problems (H13, H14, H15, Phase NB)?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Build alignment matrix: papers × problems
problems = ["H13", "H14", "H15", "Phase NB"]

alignment = {
    "1111.0931": [0, 0, 1, 0],  # Bettin-Conrey: H15 kernel
    "1601.06839": [0, 0, 1, 0],  # Auli-Bayad-Beck: H15 reciprocity
    "1211.5191": [0, 0, 0, 1],  # BCF: Phase NB (but RH assumption)
    "1401.2932": [0, 0, 1, 0],  # Farey: H15 diagnostics
    "1401.3150": [0, 0, 1, 0],  # Dirichlet poly: H15 Route 3
    "1409.1634": [0, 0, 1, 0],  # Shifted convolutions: H15
    "1501.02975": [0, 1, 0, 0],  # Mellin: H14
    "1503.05121": [0, 0, 0, 1],  # Zero distribution: Phase NB
    "1510.05087": [0, 1, 1, 0],  # Functional equations: H14/H15
    "math_0306251": [1, 0, 0, 0],  # Vasyunin: H13
    "1807.08249": [0, 0, 0, 1],  # Operator methods: Phase NB
    "2105.00004": [1, 1, 1, 1],  # RH survey: all
    "2209.10990": [0, 0, 0, 1],  # NB modern: Phase NB
    "math_9912107": [0, 1, 0, 0],  # Möbius distribution: H14
}

# Create matrix
papers_sorted = sorted(alignment.keys())
matrix = np.array([alignment[p] for p in papers_sorted]).T  # Transpose to problems × papers

# Visualize
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(papers_sorted)))
ax.set_yticks(range(len(problems)))
ax.set_xticklabels(papers_sorted, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(problems, fontsize=10)
ax.set_title('Paper-Problem Alignment Matrix', fontsize=12, fontweight='bold')

# Add text annotations
for i in range(len(problems)):
    for j in range(len(papers_sorted)):
        if matrix[i, j] > 0:
            ax.text(j, i, '✓', ha='center', va='center', color='darkred', fontweight='bold')

plt.colorbar(im, ax=ax, label='Relevance')
plt.tight_layout()
plt.savefig('/tmp/paper_problem_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Problem Coverage:")
for i, problem in enumerate(problems):
    count = np.sum(matrix[i])
    papers_involved = [p for p, val in zip(papers_sorted, matrix[i]) if val > 0]
    print(f"  {problem}: {int(count)} papers → {papers_involved}")

## 3. Citation Network and Theorem Dependencies

Map how papers cite each other and which theorems depend on which.

In [ ]:
# Citation relationships (manually curated based on mathematical knowledge)
citation_graph = {
    "1111.0931": ["2105.00004"],  # Bettin-Conrey cited in surveys
    "1601.06839": ["1111.0931"],  # Auli-Bayad-Beck builds on Bettin-Conrey
    "1211.5191": ["1111.0931", "1601.06839"],  # BCF builds on earlier work
    "2105.00004": ["1111.0931", "1211.5191", "2209.10990"],  # Survey cites core papers
    "2209.10990": ["1211.5191"],  # Modern NB treatment
}

# Theorem dependency (which theorems from which papers are used where)
theorem_deps = {
    "H15": {
        "requires": [
            ("1111.0931", "Period functional equation"),
            ("1111.0931", "Vasyunin cotangent formula"),
            ("1601.06839", "Reciprocity for cotangent sums"),
        ],
        "outputs": ["Centered bilinear residual bound O(1/log N)"]
    },
    "Phase NB": {
        "requires": [
            ("1211.5191", "Nyman-Beurling machinery"),
            ("2209.10990", "Modern NB formulation"),
            ("1503.05121", "Zero distribution"),
        ],
        "outputs": ["RH from Nyman-Beurling criterion"]
    }
}

print("\n📚 Citation Dependencies (directed edges):")
for source, targets in citation_graph.items():
    for target in targets:
        print(f"  {source} ← cites ← {target}")

print("\n🔗 Theorem Dependencies:")
for problem, deps in theorem_deps.items():
    print(f"\n  {problem}:")
    print(f"    Required theorems:")
    for arxiv, theorem in deps['requires']:
        print(f"      • {theorem} ({arxiv})")
    print(f"    Outputs to:")
    for output in deps['outputs']:
        print(f"      • {output}")

## 4. Generate Website Summary Cards

Create HTML/JSON cards for embedding in the website papers section.

In [ ]:
# Generate summary card for each core paper
def generate_paper_card(arxiv_id, paper_data):
    """Generate HTML card for a paper."""
    
    template = f"""
    <div class="paper-card" data-arxiv="{arxiv_id}">
      <h4>{paper_data.get('title', 'Untitled')}</h4>
      <p class="meta">{', '.join(paper_data.get('authors', []))} ({paper_data.get('year', '?')})</p>
      <p class="role"><strong>Role:</strong> {paper_data.get('role', 'N/A')}</p>
      {f'<p class="notes"><em>{paper_data["notes"]}</em></p>' if 'notes' in paper_data else ''}
      <a href="https://arxiv.org/abs/{arxiv_id}" target="_blank">View on arXiv →</a>
    </div>
    """
    return template.strip()

# Generate cards for core papers
core_papers = {k: v for k, v in PAPERS.items() if v.get('status') == 'core'}

print("\n📋 Website Summary Cards:")
print("\n" + "="*80)

for arxiv_id, paper_data in core_papers.items():
    card = generate_paper_card(arxiv_id, paper_data)
    print(card)
    print()

## 5. Data Visualization for Publication

Generate figures for the technical report and website.

In [ ]:
# Timeline of paper publication
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

years_counter = Counter()
for paper in PAPERS.values():
    if 'year' in paper:
        years_counter[paper['year']] += 1

years = sorted(years_counter.keys())
counts = [years_counter[y] for y in years]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(years, counts, color='steelblue', edgecolor='navy', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Year', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Papers', fontsize=11, fontweight='bold')
ax.set_title('Publication Timeline: RH Formalization Papers', fontsize=13, fontweight='bold')
ax.set_xticks(years)
ax.grid(axis='y', alpha=0.3, linestyle='--')
for year, count in zip(years, counts):
    ax.text(year, count + 0.1, str(count), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/papers_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 Papers by year:")
for year in years:
    print(f"  {year}: {years_counter[year]} paper(s)")

## 6. Export Metadata for Integration

Save structured data for website and documentation.

In [ ]:
# Export as JSON for programmatic access
export_data = {
    "metadata": {
        "generated": "2026-07-14",
        "total_papers": len(PAPERS),
        "core_papers": len([p for p in PAPERS.values() if p.get('status') == 'core']),
        "extended_collection": len([p for p in PAPERS.values() if p.get('status') == 'extended_collection'])
    },
    "papers": PAPERS,
    "alignment_by_problem": {
        "H13": [p for p, a in alignment.items() if a[0] > 0],
        "H14": [p for p, a in alignment.items() if a[1] > 0],
        "H15": [p for p, a in alignment.items() if a[2] > 0],
        "Phase NB": [p for p, a in alignment.items() if a[3] > 0],
    }
}

# Print as compact JSON
print("\n📦 Metadata export (JSON):")
print(json.dumps(export_data, indent=2)[:500] + "\n...\n(Full output available as .json file)")

# Also save to file
output_path = Path('/tmp/papers_metadata.json')
with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2)
print(f"\n✅ Exported to: {output_path}")

## 7. Dynamic Paper Addition Template

When new papers are added, use this template to extend the analysis.

In [ ]:
# Template for adding a new paper
new_paper_template = """
To add a new paper, append to the PAPERS dict:

"ARXIV_ID_HERE": {
    "arxiv_id": "ARXIV_ID_HERE",
    "authors": ["Author1", "Author2"],
    "year": YYYY,
    "title": "Paper Title Here",
    "role": "Which problem(s) it addresses (H13/H14/H15/Phase NB)",
    "key_theorems": ["Theorem 1", "Theorem 2"],
    "status": "core" or "extended_collection" or "other",
    "notes": "Any important caveats or limitations"
}

Then update:
1. alignment["ARXIV_ID"] = [H13_relevance, H14_relevance, H15_relevance, NB_relevance]
2. citation_graph["ARXIV_ID"] = [list_of_arxiv_ids_it_cites]
3. theorem_deps if it introduces key theorems
"""

print(new_paper_template)
print("\n💡 Next steps when adding papers:")
print("  1. Add to PAPERS dict above")
print("  2. Update alignment matrix")
print("  3. Update citation_graph")
print("  4. Re-run all cells to regenerate visualizations")
print("  5. Export updated metadata for website")

## 8. Lean Proof Structure Analysis (Future)

When Lean formalizations of key theorems are available, extract and visualize proof structure.

In [ ]:
# Placeholder for future Lean proof analysis

proof_analysis_notes = """
Once H14 FEFactor and H15 proofs are formalized in Lean, analyze:

1. Dependency graph: which lemmas depend on which
2. Proof length: lines of tactic code per lemma
3. Axiom usage: what axioms are actually needed
4. Tactic diversity: which tactics are most common
5. Type signature complexity: how abstract are the objects

This would give a "proof structure fingerprint" for each theorem,
useful for understanding proof difficulty and pedagogical clarity.

Example visualization: DAG of lemma dependencies with node size ~ proof length
"""

print(proof_analysis_notes)

---

## Running This Notebook

```bash
cd /path/to/riemann/project
jupyter notebook notebooks/papers_digital_humanities_analysis.ipynb
```

Outputs:
- Images: `/tmp/paper_problem_alignment.png`, `/tmp/papers_timeline.png`
- JSON metadata: `/tmp/papers_metadata.json`
- Website cards: HTML snippets ready to paste

---